<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Módulo 1: Teoría de Redes Neuronales</h1>
    <h3>Aprendizaje Automático Avanzado 2026</h3>
</div>

## Introducción

Las redes neuronales artificiales (RNA) constituyen un paradigma de computación inspirado en las <a href="https://es.wikipedia.org/wiki/Neurona">neuronas</a> biológicas y su interconexión. Las neuronas biológicas son células compuestas principalmente de tres partes: **soma** (cuerpo celular), **dendritas** (canales de entrada) y **axón** (canal de salida). De forma muy simplificada, las neuronas transmiten información mediante procesos electroquímicos: cuando una neurona recibe, a través de las dendritas, una cantidad de estímulos mayor a un cierto umbral, se despolariza y excita —a través del axón y las sinapsis— a otras neuronas conectadas a ella.

<img src="Figures/neurona.jpg"  style="display: block; margin: 0 auto;" width="70%">

## La neurona artificial

Inspirados en esta idea se concibió el modelo de <a href="https://es.wikipedia.org/wiki/Neurona_de_McCulloch-Pitts">neurona artificial</a>. Fundamentalmente, consiste en una unidad de cálculo que recibe como entrada un vector de características $\vec{x}$, cuyos valores se suman de forma ponderada mediante unos pesos $\vec{w}$. Si esta suma supera un cierto umbral $\theta$, genera un valor de salida (por ejemplo, $1$); en caso contrario, genera otro (por ejemplo, $0$). Cuando la neurona actúa de forma aislada —sin estar conectada a otras formando una red— se comporta como un **clasificador lineal**.

<img src="Figures/neurona_artificial.png" style="display: block; margin: 0 auto;" width="40%">

La expresión más básica de la neurona artificial es la siguiente:

$$
y = f(\textbf{x}) = \begin{cases} 1, & \text{si } \sum_{i=1}^{n} w_i\, x_i \geq \theta \\[6pt] 0, & \text{en caso contrario} \end{cases}
$$


## El Perceptrón como clasificador lineal

Volvamos a la definición de neurona artificial y veamos su relación con los problemas de clasificación lineal. Recordemos su expresión, pero modificándola ligeramente al mover $\theta$ a la izquierda del símbolo "mayor o igual":

$$
g(\textbf{x}) = \begin{cases} 1, & \text{si } \sum_{i=1}^{n} w_i\, x_i - \theta \geq 0 \\[6pt] 0, & \text{en caso contrario} \end{cases}
$$

Al término $-\theta$ se le suele llamar **sesgo** (*bias*) $b$, de modo que la condición se escribe como $\vec{w}\cdot\vec{x} + b \geq 0$. Podemos visualizar gráficamente la neurona de esta manera:

<img src="Figures/model.svg"  style="display: block; margin: 0 auto;" width="70%">


## Funciones de activación comunes

## 1. Sigmoide o función logística

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

Esta función "aplasta" (*squash*) cualquier entrada al rango $[0, 1]$, por lo que su salida puede interpretarse como una probabilidad. Su derivada es $\sigma'(x) = \sigma(x)\,(1 - \sigma(x))$.


In [ ]:
import os

# Evita el conflicto de múltiples runtimes OpenMP (MKL + libgomp) en Windows
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

def grad_sigmoid(x):
    return sigmoid(x)*(1-sigmoid(x))


In [ ]:
x = np.arange(-10, 10, 0.1)
plt.figure(figsize=(8, 6))
plt.plot(x, sigmoid(x), label='Sigmoide $\\sigma(x)$', c='blue', linewidth=3)
plt.plot(x, grad_sigmoid(x), label="Derivada $\\sigma'(x)$", c='green', linewidth=3)
plt.legend(loc='upper left')
plt.xlabel('x')
plt.ylabel('$\\sigma(x)$')
plt.grid()
plt.ylim([-0.5, 1.5])
plt.title('Sigmoide')
plt.show()

## Problemas de la función sigmoide

#### 1. Desvanecimiento del gradiente (*vanishing gradient*)
En la figura anterior, cuando la salida está cerca de 0 o de 1, la derivada es casi cero. Esto significa que, durante la retropropagación, los pesos se actualizarán muy lentamente (de forma casi insignificante), por lo que el aprendizaje será casi nulo y los pesos apenas cambiarán respecto a su valor inicial.

Además, el valor máximo de la derivada de la sigmoide es $0.25$. Por lo tanto, cada gradiente se reduce al menos al 25 % de su valor (en el peor caso, a 0) y, en una red profunda, se pierde cada vez más señal. Los gradientes que llegan a las capas superficiales (cercanas a la entrada) resultan demasiado pequeños para actualizar los pesos de forma efectiva.

#### 2. No está centrada en cero (*not zero-centered*)
En el algoritmo de propagación:

$$f=\sum w_ix_i+b \qquad \frac{df}{dw_i}=x_i \qquad \frac{dL}{dw_i}=\frac{dL}{df}\frac{df}{dw_i}=\frac{dL}{df}x_i$$

Como $x_i > 0$, el gradiente $\dfrac{dL}{dw_i}$ siempre tiene el mismo signo que $\dfrac{dL}{df}$ (todos positivos o todos negativos). Por lo tanto, si un peso debe actualizarse en sentido positivo y otro en sentido negativo, no podrá hacerse en el mismo paso, lo que ralentiza la convergencia.

#### 3. Costosa de calcular (*computationally expensive*)
El cálculo de la exponencial de la sigmoide es computacionalmente costoso.


## 2. Tangente Hiperbólica

$$f = tanh(x)$$

La salida se limita a [-1, 1]. Esta función es casi la misma que la función sigmoide, pero está centrada en cero, por lo tanto, es mejor que la función sigmoide.

Además, tanh es solo una versión escalada de la sigmoide:

$$tanh(x) = 2\sigma(2x) - 1$$

In [ ]:
def grad_tanh(x):
    return 1 - np.tanh(x) ** 2

x = np.arange(-10, 10, 0.1)
plt.figure(figsize=(8, 6))
plt.plot(x, np.tanh(x), label='Tangente hiperbólica $\\tanh(x)$', c='blue', linewidth=3)
plt.plot(x, grad_tanh(x), label="Derivada $\\tanh'(x)$", c='green', linewidth=3)
plt.legend(loc='upper left')
plt.xlabel('x')
plt.ylabel('$\\tanh(x)$')
plt.grid()
plt.ylim([-1.5, 1.5])
plt.title('Tangente Hiperbólica')
plt.show()

## 3. Rectified Linear Unit (ReLU)

$$f(x) = max(0, x)$$

ReLU es una de las funciones de activación más utilizadas que tiene muchas ventajas:

#### 1. Resuelve el problema del vanishing gradient 
ReLU no sufre el problema del gradiente de fuga ya que las pendientes son arbitrarias con respecto a las entradas

#### 2. Computacionalmente barato y simple de implementar
ReLU no involucra ninguna operación matemática y simplemente reemplaza x < 0 con 0.

### Problema con la ReLU

#### Problema Dying ReLU  

Debido a la naturaleza de ReLU, si un gradiente se propaga a la entrada, la red puede llegar a un estado en el que el sesgo sea muy bajo (negativo) y, por lo tanto, la salida de ReLU será 0 sin importar la entrada. Y dado que el gradiente de 0 es 0, la red no se recuperará de este estado y permanecerá siempre igual.


In [ ]:
def relu(x):
    return np.maximum(0, x)

def grad_relu(x):
    return np.where(x > 0, 1, 0)

x = np.arange(-3, 3, 0.01)

plt.figure(figsize=(8, 6))
plt.plot(x, relu(x), label='ReLU', c='blue', linewidth=3)
plt.plot(x, grad_relu(x), label='Derivada de ReLU', c='green', linewidth=3)
plt.legend(loc='upper left')
plt.xlabel('x')
plt.ylabel('ReLU(x)')
plt.grid()
plt.ylim([-0.5, 3])
plt.title('ReLU')
plt.show()

## 4. Leaky ReLU

$$f(x) = \max(\alpha x,\, x)$$

Leaky ReLU busca resolver el problema de la *Dying ReLU* asignando una pequeña pendiente $\alpha$ (por ejemplo, $0.1$) a los valores negativos, en lugar de anularlos por completo. Así, el gradiente sobre los valores negativos es $\alpha$ (distinto de cero) y la red puede recuperarse.


In [ ]:
def leaky_relu(x, alpha=0.1):
    return np.maximum(alpha * x, x)

def grad_leaky_relu(x, alpha=0.1):
    return np.where(x > 0, 1, alpha)

x = np.arange(-3, 3, 0.01)
plt.figure(figsize=(8, 6))
plt.plot(x, leaky_relu(x), label='Leaky ReLU', c='blue', linewidth=3)
plt.plot(x, grad_leaky_relu(x), label='Derivada de Leaky ReLU', c='green', linewidth=3)
plt.legend(loc='upper left')
plt.xlabel('x')
plt.ylabel('Leaky ReLU(x)')
plt.grid()
plt.ylim([-0.5, 3])
plt.title('Leaky ReLU')
plt.show()

## 5. Softmax

La función **softmax** se utiliza en la **capa de salida** de los problemas de clasificación multiclase. Convierte un vector de $K$ valores reales (los *logits* $z$) en una distribución de probabilidad: todos los valores quedan en el rango $(0, 1)$ y suman 1.

$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

Puede verse como una generalización de la sigmoide a varias clases. La clase predicha es la de mayor probabilidad ($\arg\max$). En la práctica se resta el máximo antes de exponenciar ($e^{z_i - \max(z)}$) por **estabilidad numérica**, evitando desbordamientos.


In [ ]:
def softmax(z):
    e = np.exp(z - np.max(z))  # resta del máximo por estabilidad numérica
    return e / e.sum()

z = np.array([2.0, 1.0, 0.1])
p = softmax(z)
print("Logits z      :", z)
print("softmax(z)    :", np.round(p, 4))
print("Suma          :", round(p.sum(), 4))
print("Clase predicha:", np.argmax(p))

plt.figure(figsize=(6, 4))
plt.bar([f'clase {i}' for i in range(len(z))], p, color='steelblue')
plt.ylabel('Probabilidad')
plt.title('Salida de softmax (suma = 1)')
plt.ylim([0, 1])
plt.show()


#### ¿Qué función de activación usar?

| Función | Rango | Uso típico | Nota |
|---|---|---|---|
| **Sigmoide** | $(0, 1)$ | Salida binaria (probabilidad) | Sufre *vanishing gradient* |
| **Tanh** | $(-1, 1)$ | Capas ocultas (centrada en 0) | Mejor que sigmoide |
| **ReLU** | $[0, \infty)$ | Capas ocultas (por defecto) | Rápida; riesgo de *dying ReLU* |
| **Leaky ReLU** | $(-\infty, \infty)$ | Capas ocultas | Evita neuronas muertas |
| **Softmax** | $(0, 1)$, suma 1 | Capa de salida multiclase | Devuelve probabilidades |


### Regla de aprendizaje

- En las RNA se considera que el conocimiento se encuentra representado en los pesos de las conexiones.

- El proceso de aprendizaje se basa en cambios en estos pesos.

### Formas de conexión entre neuronas

- Las salidas de las neuronas se convierten en entradas de otras neuronas.
- Cuando ninguna salida de las neuronas es entrada de neuronas del mismo nivel o de niveles precedentes, la red se describe como propagación hacia adelante (feedforward).

- En caso contrario la red se describe como propagación hacia atrás (feedback).

### Características de las RNA

>Topología

- Número de capas.

- Número de neuronas por capa.

- Tipo de conexiones. Normalmente, todas las neuronas de una capa reciben señales de la capa anterior (más cercana a la entrada) y envían su salida a las neuronas de la capa posterior (más cercana a la salida de la red).

>Tipos de aprendizaje

### Redes feedforward

La más conocidas son:
- perceptron
- Adaline
- Madaline
- Backpropagation

útilies en aplicaciones de reconocimiento o clasificación de patrones.

### Cómo funciona una red neuronal

<img src="Figures/red_pesos.png"  style="display: block; margin: 0 auto;" width="50%">

Las redes neuronales se modelan como colecciones de neuronas que están conectadas en un gráfico acíclico. En otras palabras, las salidas de algunas neuronas pueden convertirse en entradas para otras neuronas. Los ciclos no están permitidos ya que eso implicaría un bucle infinito en el paso hacia adelante de una red. Los modelos de redes neuronales a menudo se organizan en distintas capas de neuronas.

Para las redes neuronales regulares, el tipo de capa más común es el **fully connected layer** en donde las neuronas entre dos capas adyacentes están completamente conectadas por pares, pero las neuronas dentro de una sola capa no comparten conexiones

<img src="Figures/RAN_layers.png"  style="display: block; margin: 0 auto;" width="80%">

**Capa de salida**. A diferencia de todas las capas de una red neuronal, las neuronas comúnmente no tienen una función de activación (o se puede pensar que tienen una función de activación lineal). Esto se debe a que la última capa de salida se
toma para representar los scores de clase (por ejemplo, en la clasificación), o pueden ser números reales (por ejemplo, en regresión)

### Cómo aprende una red neuronal

>Por lo general, un modelo de red neuronal se entrena utilizando el Pesos y algoritmo de optimización de descenso de gradiente estocástico se actualizan usando el algoritmo backpropagation del error

>El "gradiente" en el algoritmo de gradiente descendente se refiere a un gradiente de error. El modelo con un conjunto dado de pesos se utiliza para hacer predicciones y se calcula el error de esas predicciones.

>El algoritmo de gradiente descendente busca cambiar los pesos para que la próxima iteración de tal forma que se reduzca el error, lo que significa que el algoritmo de optimización está navegando hacia abajo en el gradiente (o pendiente) del error.

<img src="Figures/gradient_descent.png"  style="display: block; margin: 0 auto;" width="80%">

#### Función de coste (*loss function*)

La función de coste ($l$), también llamada función de pérdida (*loss*) o de costo, cuantifica la distancia entre el valor real y el valor predicho por la red; en otras palabras, mide **cuánto se equivoca** la red al hacer predicciones. En la mayoría de los casos devuelve valores positivos: cuanto más próximo a cero, mejores son las predicciones (menor error), siendo cero cuando las predicciones coinciden exactamente con el valor real.

La función de coste puede calcularse para una única observación o para un conjunto de datos (normalmente promediando el valor de todas las observaciones). El segundo caso es el que se utiliza para dirigir el entrenamiento de los modelos.

Dependiendo del tipo de problema —regresión o clasificación— se utiliza una función de coste distinta. En regresión, las más habituales son el **error cuadrático medio** y el **error absoluto medio**. En clasificación suele emplearse la función *log loss*, también llamada *logistic loss* o *cross-entropy loss*.

#### Error cuadrático medio (MSE)

El error cuadrático medio (*mean squared error*, MSE) es, con diferencia, la función de coste más utilizada en problemas de regresión. Para una observación $i$, el error cuadrático es la diferencia al cuadrado entre el valor predicho $\hat{y}$ y el valor real $y$:

$$l^{(i)}(w,b)=\left(\hat{y}^{(i)} - y^{(i)}\right)^2$$

Las funciones de coste suelen escribirse como $l(w,b)$ para indicar que su valor depende de los pesos y el sesgo del modelo, ya que son estos los que determinan las predicciones. Con frecuencia se multiplica por $\tfrac{1}{2}$, por conveniencia matemática, para simplificar el cálculo de su derivada:

$$l^{(i)}(w,b)=\frac{1}{2}\left(\hat{y}^{(i)} - y^{(i)}\right)^2$$

Para cuantificar el error sobre todo un conjunto de datos (por ejemplo, el de entrenamiento), se promedia el error de las $n$ observaciones:

$$L(w,b)=\frac{1}{n}\sum_{i=1}^n l^{(i)}(w,b)= \frac{1}{n}\sum_{i=1}^n \left(\hat{y}^{(i)} - y^{(i)}\right)^2$$

Cuando un modelo se entrena con MSE, aprende a predecir la **media** de la variable respuesta.

#### Error absoluto medio (MAE)

El error absoluto medio (*mean absolute error*, MAE) promedia el valor absoluto de los errores:

$$L(w,b)= \frac{1}{n}\sum_{i=1}^n \left|\hat{y}^{(i)} - y^{(i)}\right|$$

El MAE es más **robusto frente a valores atípicos** (*outliers*) que el MSE, por lo que el entrenamiento se ve menos influenciado por datos anómalos. Cuando un modelo se entrena con MAE, aprende a predecir la **mediana** de la variable respuesta.

#### Log loss / cross-entropy loss

En clasificación, la capa de salida suele usar la función **softmax**, que devuelve valores interpretables como la probabilidad de que la observación pertenezca a cada clase.

En clasificación **binaria**, donde la variable respuesta es 1 o 0 y $p = \Pr(y=1)$, la función de coste se define como:

$$L_{\log}(y,p)=-\log\Pr(y\mid p)=-\big(y\log(p)+(1-y)\log(1-p)\big)$$

Para clasificación con **más de dos clases** ($K$ clases y $N$ observaciones), se generaliza a:

$$L_{\log}(Y,P)=-\frac{1}{N} \sum_{i=0}^{N-1} \sum_{k=0}^{K-1} y_{i,k}\,\log p_{i,k}$$

En ambos casos, minimizar esta función equivale a que la probabilidad predicha para la clase correcta tienda a 1, y a 0 en las demás. Según el campo, se le conoce como *log loss*, *logistic loss* o *cross-entropy loss*, pero todos hacen referencia a lo mismo.

### Múltiples capas

El modelo de red neuronal con una única capa (*single-layer perceptron*), aunque supuso un gran avance, solo es capaz de aprender patrones sencillos. Para superar esta limitación, se descubrió que combinando **múltiples capas ocultas** la red puede aprender relaciones mucho más complejas entre los predictores y la variable respuesta. A esta estructura se le conoce como **perceptrón multicapa** (*multilayer perceptron*, MLP) y puede considerarse el primer modelo de *deep learning*.

Cada neurona está conectada a todas las neuronas de la capa anterior y de la posterior. Aunque no es estrictamente necesario, todas las neuronas de una misma capa suelen emplear la misma función de activación. Combinando múltiples capas ocultas y funciones de activación **no lineales**, un MLP puede aproximar prácticamente cualquier función: de hecho, está demostrado que, con suficientes neuronas, un MLP es un **aproximador universal**.

Red neuronal *feed-forward* (perceptrón multicapa):
<img src="Figures/multilayer.png"  style="display: block; margin: 0 auto;" width="60%">

### Entrenamiento

El entrenamiento de una red neuronal consiste en ajustar los pesos y sesgos de tal forma que las predicciones tengan el menor error posible. La idea intuitiva es la siguiente:

1. Iniciar la red con valores aleatorios de pesos y sesgos.
2. Para cada observación de entrenamiento $(X, y)$, calcular el error de la predicción y promediar los errores de todas las observaciones.
3. Identificar la responsabilidad de cada peso y sesgo en el error.
4. Modificar ligeramente los pesos y sesgos (de forma proporcional a su responsabilidad) en la dirección que reduce el error.
5. Repetir los pasos 2–4 hasta que la red sea suficientemente buena.

Aunque la idea parece sencilla, implementarla requiere combinar el algoritmo de **retropropagación** (*backpropagation*) y la optimización por **descenso de gradiente** (*gradient descent*).

### Backpropagation

*Backpropagation* es el algoritmo que permite cuantificar la influencia de cada peso y sesgo en las predicciones. Para ello, usa la **regla de la cadena** (*chain rule*) para calcular el **gradiente**, que es el vector de derivadas parciales del error respecto a cada parámetro. La derivada parcial del error respecto a un parámetro mide cuánta "responsabilidad" tuvo ese parámetro en el error cometido, lo que indica qué pesos hay que modificar para mejorar la red.

### Descenso de gradiente

El descenso de gradiente es un algoritmo de optimización que minimiza una función actualizando sus parámetros en la dirección del **negativo de su gradiente**. Aplicado a las redes, permite ir actualizando pesos y sesgos para reducir el error.

Dado que calcular el error para todas las observaciones en cada iteración puede ser muy costoso, existe una alternativa llamada **descenso de gradiente estocástico** (*stochastic gradient descent*, SGD). Consiste en dividir el conjunto de entrenamiento en **lotes** (*minibatch*) y actualizar los parámetros con cada uno. Una ronda completa sobre todos los lotes se llama **época** (*epoch*); el número de épocas es el número de veces que la red ve cada ejemplo de entrenamiento.

### Preprocesado

Al entrenar redes neuronales es necesario aplicar, al menos, dos tipos de transformación a los datos.

#### One-hot encoding de variables categóricas

Consiste en crear nuevas variables *dummy* para cada nivel de las variables cualitativas. Por ejemplo, una variable `color` con niveles rojo, verde y azul se convierte en tres variables (`color_rojo`, `color_verde`, `color_azul`), todas con valor 0 excepto la que coincide con la observación, que toma el valor 1.

#### Estandarización y escalado de variables numéricas

La escala y la varianza de los predictores numéricos influyen mucho en el modelo: si no se igualan, los predictores con mayor escala o varianza dominarán aunque no sean los más informativos. Las estrategias principales son:

- **Centrado**: restar a cada valor la media de su predictor, de modo que todos queden centrados en torno al origen (media cero).
- **Normalización (estandarización)**: transformar los datos para que todos los predictores estén aproximadamente en la misma escala.
    - **Z-score** (`StandardScaler`): dividir cada predictor centrado entre su desviación típica, obteniendo media 0 y desviación 1.
    - **Min-max** (`MinMaxScaler`): reescalar los datos al rango $[0, 1]$.

## Hiperparámetros

La gran flexibilidad de las redes neuronales es un arma de doble filo: pueden aprender relaciones muy complejas, pero sufren fácilmente **sobreajuste** (*overfitting*), lo que les impide predecir bien nuevas observaciones. Para minimizarlo hay que configurar adecuadamente sus hiperparámetros. Algunos de los más importantes son:

### Número y tamaño de capas

La arquitectura (número de capas y de neuronas por capa) determina la complejidad del modelo y su capacidad de aprendizaje. La capa de **entrada** tiene tantas neuronas como predictores; la de **salida** tiene una neurona en regresión y tantas como clases en clasificación. El usuario suele especificar solo el número y tamaño de las capas **ocultas**. A más neuronas y capas, mayor complejidad aprendible, pero también más parámetros y mayor tiempo de entrenamiento.

### Learning rate

El *learning rate* (ratio de aprendizaje) establece cómo de rápido cambian los parámetros durante la optimización. Es uno de los más difíciles de ajustar. Si es muy **grande**, la optimización puede saltar de una región a otra sin converger; si es muy **pequeño**, el entrenamiento puede tardar demasiado. Recomendaciones heurísticas:

- Usar un *learning rate* lo más pequeño posible, siempre que el tiempo de entrenamiento sea aceptable.
- No usar un valor constante: por lo general, valores mayores al inicio y menores al final.

### Algoritmo de optimización

El descenso de gradiente (y su variante estocástica) fueron los primeros métodos usados para entrenar redes. Sin embargo, a medida que las redes crecen, muchas regiones del espacio de búsqueda tienen gradiente próximo a cero, lo que estanca la optimización. Por ello se desarrollaron variantes que **adaptan el learning rate** según el gradiente. Recomendaciones habituales:

- Conjuntos de datos **pequeños**: `l-bfgs`.
- Conjuntos de datos **grandes**: `adam` o `rmsprop`.

Una descripción detallada puede encontrarse en el libro gratuito [Dive into Deep Learning](http://d2l.ai/chapter_optimization/index.html).

### Regularización

Los métodos de regularización buscan reducir el **sobreajuste**. Como las redes neuronales suelen estar sobreparametrizadas, la regularización es fundamental. Las técnicas más destacadas son la regularización L1/L2 (*weight decay*) y el *dropout*.

#### Penalización L1 y L2

El objetivo de la penalización L1 y L2 (esta última también conocida como *weight decay*) es evitar que los pesos tomen valores excesivamente elevados. Así se impide que unas pocas neuronas dominen la red y se fuerza a que las características poco informativas (ruido) tengan pesos próximos o iguales a cero.

#### Dropout

Consiste en **desactivar aleatoriamente** una fracción de neuronas durante el entrenamiento: en cada iteración se ponen a cero los pesos de una fracción aleatoria de neuronas por capa. Descrito por Srivastava et al. (2014), se ha convertido en un estándar. El porcentaje de neuronas desactivadas por capa (*dropout rate*) suele estar entre 0.2 y 0.5.


### Perceptrón

El **perceptrón** es un algoritmo simple que, dado un vector de entrada $\textbf{x}$ de $m$ valores $(x_1, x_2, \dots, x_m)$ —llamados características de entrada—, genera un $1$ ("sí") o un $0$ ("no"). Matemáticamente definimos la función:

$$
f(\textbf{x}) = \begin{cases} 1, & \text{si } \sum_{i=1}^{n} w_i\, x_i + b \geq 0 \\[6pt] 0, & \text{en caso contrario} \end{cases}
$$

Donde $\textbf{w}=(w_1, w_2, \dots, w_n)$ es el vector de pesos, $\textbf{w}\cdot\textbf{x} = \sum_{i=1}^{n} w_i\, x_i$ es el producto escalar y $b$ es el **sesgo** (*bias*). La expresión $\textbf{w}\cdot\textbf{x} + b = 0$ define un **hiperplano** que separa el espacio en dos regiones y cuya posición cambia según los valores de $\textbf{w}$ y $b$.

> Un hiperplano es un subespacio cuya dimensión es uno menos que la de su espacio ambiente (una recta en 2D, un plano en 3D, etc.).

<img src="Figures/hiperplane.png"  style="display: block; margin: 0 auto;" width="40%">

¡Es un algoritmo muy simple pero efectivo! Por ejemplo, dadas tres características de entrada —las cantidades de rojo, verde y azul de un color—, el perceptrón podría decidir si el color es blanco o no.

Ten en cuenta que el perceptrón no puede expresar una respuesta "tal vez": responde "sí" (1) o "no" (0), según cómo se definan $\textbf{w}$ y $b$. Determinar esos valores es el proceso de **entrenamiento**, que se aborda en las siguientes secciones.


## Construir una RN con TensorFlow 2.0

Hay cuatro partes principales:

>**Construcción del modelo**： `tf.keras.Model` con `tf.keras.layers`

>**Función de pérdida de modelo**： `tf.keras.losses`

>**Optimizador de modelos**： `tf.keras.optimizer`

>**Evaluación del modelo：** `tf.keras.metrics`

Hay tres formas de crear un modelo en `tf.keras`: API secuencial, API funcional y subclases de modelos. Aquí, usaremos el más simple, `Sequential()`.

El siguiente fragmento de código define una sola capa con 10 neuronas artificiales que espera 784 variables de entrada (también conocidas como características). Hay que tener en cuenta que la red es "densa", lo que significa que cada neurona en una capa está conectada a todas las neuronas ubicadas en la capa anterior y a todas las neuronas en la capa siguiente:


In [ ]:
import tensorflow as tf
from tensorflow import keras

NB_CLASSES = 10
RESHAPED = 784
model = tf.keras.models.Sequential()
model.add(keras.layers.Dense(NB_CLASSES,
                             input_shape = (RESHAPED,),
                             kernel_initializer='zeros',
                             name='dense_layer',
                             activation='softmax'))

Cada neurona se puede inicializar con pesos específicos a través del parámetro `kernel_initializer`. Hay algunas opciones, las más comunes se enumeran a continuación:

>`random_uniform`: Los pesos se inicializan en pequeños valores uniformemente aleatorios en el rango de -0,05 a 0,05.

>`random_normal`: Los pesos se inicializan de acuerdo con una distribución gaussiana, con media cero y una pequeña desviación estándar de 0,05. Para aquellos de ustedes que no están familiarizados con la distribución gaussiana, piensen en una forma de "curva de campana" simétrica.

>`zero`: Todos los pesos se inicializan a cero.

Una lista completa de como inicializar está en: https://www.tensorflow.org/api_docs/python/tf/keras/initializers.

## Ejemplo de un Perceptrón Multicapa (reconocimiento de dígitos escritos a mano)

En esta sección construiremos una red que puede reconocer números escritos a mano. Para lograr este objetivo, utilizaremos [`MNIST`](http://yann.lecun.com/exdb/mnist/), una base de datos de dígitos escritos a mano compuesta por un conjunto de entrenamiento de 60 000 ejemplos y un conjunto de prueba de 10 000 ejemplos Los ejemplos de entrenamiento son anotados por humanos con la respuesta correcta. Por ejemplo, si el dígito escrito a mano es el número "3", entonces 3 es simplemente la etiqueta asociada con ese ejemplo.

### One-hot encoding (OHE)

Vamos a usar OHE como una herramienta simple para codificar información utilizada dentro de redes neuronales. En muchas aplicaciones es conveniente transformar características categóricas (no numéricas) en variables numéricas. Por ejemplo, la característica categórica "dígito" con valor d en [0 – 9] se puede codificar en un vector binario con 10 posiciones, que siempre tiene valor 0 excepto la posición d-ésima donde está presente un 1.

Por ejemplo, el dígito 3 se puede codificar como [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]. Este tipo de representación se denomina codificación one-hot, o a veces simplemente one-hot, y es muy común en la minería de datos cuando el algoritmo de aprendizaje se especializa en el manejo de funciones numéricas.


### 1. Definición de una red neuronal simple en TensorFlow 2.0

Usaremos TensorFlow 2.0 para definir una red que reconozca los dígitos escritos a mano del MNIST. Empezamos con una red neuronal muy simple y luego la mejoramos progresivamente.

> Siguiendo el estilo de Keras, TensorFlow 2.0 proporciona [bibliotecas adecuadas](https://www.tensorflow.org/api_docs/python/tf/keras/datasets) para cargar el conjunto de datos y dividirlo en conjuntos de entrenamiento, `X_train`, que se usan para el entrenamiento de nuestra red y conjuntos de prueba, `X_test`, utilizados para evaluar el rendimiento. Los datos se convierten en `float32` para usar una precisión de 32 bits al entrenar una red neuronal y se normalizan al rango [0,1]. Además, cargamos las etiquetas verdaderas en `Y_train` y `Y_test` respectivamente, y realizamos una codificación one-hot en ellas. 



In [ ]:
import numpy as np

In [ ]:
EPOCHS = 20              # número de veces que la red ve todo el conjunto de entrenamiento
BATCH_SIZE = 128         # muestras procesadas antes de cada actualización de pesos
VERBOSE = 1
NB_CLASSES = 10          # número de clases de salida (dígitos 0-9)
N_HIDDEN = 128           # neuronas por capa oculta
VALIDATION_SPLIT = 0.2   # fracción del entrenamiento reservada para validación (20%)

Intuitivamente, `EPOCH` define cuánto debe durar el entrenamiento, `BATCH_SIZE` es la cantidad de muestras que alimenta a su red por cada iteración, y `VALIDATION` es la cantidad de datos reservados para verificar o probar la validez del entrenamiento. Veamos nuestro primer fragmento de código de una red neuronal en TensorFlow.


In [ ]:
#Cargar el dataset MNIST
mnist = keras.datasets.mnist
# Obtener los datos de entrenamiento y testeo
(X_train, y_train), (X_test, y_test) = mnist.load_data() # 60,000 muestras, con 28x28 valores
X_train = X_train.reshape(60000,784)
X_test = X_test.reshape(10000,784)
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

#Normalizar las entradas en el rango de [0,1]
X_train/=255
X_test/=255

#Aplicar onehot encoding a la variable de salida
y_train = tf.keras.utils.to_categorical(y_train, NB_CLASSES)
y_test = tf.keras.utils.to_categorical(y_test, NB_CLASSES)

In [ ]:
# Verificar que los datos quedaron normalizados en [0, 1]


In [ ]:
# Cada imagen de 28x28 píxeles se aplana en un vector de 784 valores


In [ ]:
# Las etiquetas ya están en formato one-hot (cada fila tiene un 1 en la clase correcta)


#### Visualizar algunos ejemplos del conjunto MNIST

Antes de entrenar, conviene ver con qué datos trabaja la red. Cada imagen es de $28\times28$ píxeles (aquí aplanada a 784 valores y normalizada). Mostramos una cuadrícula con su etiqueta (obtenida del vector one-hot con `argmax`).

In [ ]:
# Mostrar 10 dígitos de entrenamiento con su etiqueta
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(X_train[i].reshape(28, 28), cmap='gray')
    plt.title(f'Etiqueta: {np.argmax(y_train[i])}')
    plt.axis('off')
plt.tight_layout()
plt.show()


Se puede ver en el código anterior que la capa de entrada tiene una neurona asociada a cada píxel de la imagen para un total de **28*28=784 neuronas**, una para cada píxel en las imágenes MNIST.

Normalmente, los valores asociados con cada píxel se normalizan en el rango [0,1] (lo que significa que la intensidad de cada píxel se divide por 255, el valor máximo de intensidad). La salida puede ser una de diez clases, con una clase para cada dígito.

La capa final es una sola neurona con función de activación "`softmax`", que es una generalización de la función sigmoidea. Como se mencionó anteriormente, la salida de una función sigmoidea está en el rango (0, 1). De manera similar, un softmax "aplasta" un vector K-dimensional de valores reales arbitrarios en un vector K-dimensional de valores reales en el rango (0, 1), de modo que todos suman 1. En nuestro caso, agrega 10 respuestas proporcionadas por la capa anterior con 10 neuronas.

In [ ]:
#Crear el modelo de red neuronal multicapa


Una vez definido el modelo, hay que **compilarlo** para que pueda ser ejecutado por TensorFlow 2.0. En la compilación se eligen tres cosas:

1. Un **optimizador**: el algoritmo que actualiza los pesos durante el entrenamiento (lista completa [aquí](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers)).
2. Una **función de pérdida** (u objetivo): la que el optimizador minimiza al navegar por el espacio de pesos.
3. Una o varias **métricas** para evaluar el modelo entrenado.

#### Funciones objetivo (pérdida) comunes

- **`MSE`** (error cuadrático medio): entre predicciones $d$ y valores observados $y$, $MSE=\frac{1}{n}\sum_{i=1}^n(d_i-y_i)^2$. Al elevar al cuadrado, penaliza más los errores grandes e ignora el signo.
- **`binary_crossentropy`** (pérdida logarítmica binaria): si el modelo predice $p$ y el objetivo es $c$, entonces $\mathcal{L}=-c\ln(p)-(1-c)\ln(1-p)$. Adecuada para etiquetas binarias.
- **`categorical_crossentropy`** (pérdida logarítmica multiclase): compara la distribución predicha con la real (probabilidad 1 en la clase verdadera y 0 en las demás): $\mathcal{L}(c,p)=-\sum_i c_i\ln(p_i)$. Es la opción por defecto junto con la activación **softmax** (lista completa [aquí](https://www.tensorflow.org/api_docs/python/tf/keras/losses)).

#### Métricas comunes

- **`Accuracy`**: proporción de predicciones correctas respecto al total.
- **`Precision`**: de los elementos seleccionados, cuántos son relevantes.
- **`Recall`**: de los elementos relevantes, cuántos fueron seleccionados.

> **Métrica vs. función de pérdida:** la **pérdida** es la función que el optimizador **minimiza** para entrenar la red; la **métrica** solo sirve para **juzgar** el rendimiento y no interviene en la optimización. A veces sería ideal optimizar directamente una métrica, pero muchas no son diferenciables respecto a sus entradas, lo que lo impide. Lista completa de métricas [aquí](https://www.tensorflow.org/api_docs/python/tf/keras/metrics).

Al compilar un modelo en TensorFlow 2.0 se seleccionan el optimizador, la función de pérdida y la métrica juntos:


In [ ]:
#Compilar el modelo


>**epochs** es la cantidad de veces que el modelo se expone al conjunto de entrenamiento. En cada iteración, el optimizador intenta ajustar los pesos para que la función objetivo se minimice.

>**batch_size** es el número de instancias de entrenamiento observadas antes de que el optimizador realice una actualización de peso; suele haber muchos lotes por época.

Entrenar un modelo en TensorFlow 2.0 es muy simple:

In [ ]:
#Entrenar el modelo




Hay que tomar en cuenta que hemos reservado parte del conjunto de entrenamiento para la validación. La idea clave es que reservamos una parte de los datos de entrenamiento para medir el rendimiento en la validación durante el entrenamiento. Esta es una buena práctica a seguir para cualquier tarea de aprendizaje automático. 

Una vez que se entrena el modelo, podemos evaluarlo en el conjunto de prueba que contiene nuevos ejemplos nunca vistos por el modelo durante la fase de entrenamiento.

En TensorFlow 2.0, podemos usar el método de evaluación (X_test, Y_test) para calcular test_loss y test_acc:

In [ ]:
# Evaluar el modelo en el conjunto de prueba


### 2. Mejorando la red simple en TensorFlow 2.0 con capas ocultas

Es un buen punto de partida, pero podemos mejorarlo. Veamos cómo.

Una mejora inicial es agregar capas adicionales a nuestra red porque estas neuronas adicionales podrían ayudarla intuitivamente a aprender patrones más complejos en los datos de entrenamiento. En otras palabras, las capas adicionales agregan más parámetros, lo que potencialmente permite que un modelo memorice patrones más complejos. Entonces, después de la capa de entrada, tenemos una primera capa densa con N_HIDDEN neuronas y una función de activación "ReLU". Esta capa adicional se considera oculta porque no está conectada directamente ni con la entrada ni con la salida. Después de la primera capa oculta, tenemos una segunda capa oculta nuevamente con N_HIDDEN neuronas seguida de una capa de salida con 10 neuronas, cada una de las cuales se disparará cuando se reconozca el dígito relativo. El siguiente código define esta nueva red:


In [ ]:
EPOCHS = 20
BATCH_SIZE = 128
VERBOSE = 1
NB_CLASSES = 10          # número de clases de salida (dígitos 0-9)
N_HIDDEN = 128           # neuronas por capa oculta
VALIDATION_SPLIT = 0.2   # fracción del entrenamiento reservada para validación (20%)

# Crear el modelo con capas ocultas

# Capa de entrada + primera capa oculta (ReLU)

# Segunda capa oculta (ReLU)

# Capa de salida (softmax)

# Resumen del modelo


In [ ]:
# Compilar el modelo

# Entrenar el modelo (guardamos el historial para graficar las curvas)


In [ ]:
# Evaluar el modelo en el conjunto de prueba


### 3. Mejorar aún más la red simple en TensorFlow con Dropout

Una segunda mejora es muy simple. Decidimos descartar aleatoriamente, con la probabilidad DROPOUT, algunos de los valores propagados dentro de nuestra densa red interna de capas ocultas durante el entrenamiento. En el aprendizaje automático, esta es una forma bien conocida de regularización. Sorprendentemente, esta idea de descartar aleatoriamente algunos valores puede mejorar nuestro rendimiento. La idea detrás de esta mejora es que la caída aleatoria obliga a la red a aprender patrones redundantes que son útiles para una mejor generalización:

In [ ]:
EPOCHS = 20
BATCH_SIZE = 128
VERBOSE = 1
NB_CLASSES = 10
N_HIDDEN = 128
VALIDATION_SPLIT = 0.2
DROPOUT = 0.3            # fracción de neuronas desactivadas en cada capa de dropout

# Crear el modelo con capas ocultas y dropout

# Compilar y entrenar (guardamos el historial para graficar las curvas)

# Evaluar el modelo


## Análisis de resultados

### Curvas de entrenamiento

Graficar la **pérdida** y la **precisión** de entrenamiento frente a las de validación (época a época) permite diagnosticar el comportamiento del modelo. Cuando la curva de entrenamiento sigue mejorando mientras la de validación se estanca o empeora, es señal de **sobreajuste** (*overfitting*). Comparamos el modelo con capas ocultas (`history2`) y el modelo con *dropout* (`history3`).

In [ ]:
def graficar_historial(history, titulo):
    h = history.history
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(h['loss'], label='entrenamiento')
    ax1.plot(h['val_loss'], label='validación')
    ax1.set_title(f'{titulo} — Pérdida')
    ax1.set_xlabel('Época'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(h['accuracy'], label='entrenamiento')
    ax2.plot(h['val_accuracy'], label='validación')
    ax2.set_title(f'{titulo} — Precisión')
    ax2.set_xlabel('Época'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

graficar_historial(history2, 'Capas ocultas (sin dropout)')
graficar_historial(history3, 'Con dropout')

### Matriz de confusión y reporte de clasificación

La **matriz de confusión** muestra cuántas veces cada dígito real (filas) fue clasificado como cada dígito predicho (columnas). Los valores de la diagonal son los aciertos; los de fuera, los errores. El **reporte de clasificación** resume *precision*, *recall* y *F1* por clase. Usamos el modelo con capas ocultas (`model2`).

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Predicciones del modelo con capas ocultas
y_prob = model2.predict(X_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
plt.xticks(range(10)); plt.yticks(range(10))
plt.xlabel('Predicción'); plt.ylabel('Etiqueta real')
plt.title('Matriz de confusión (model2)')
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, digits=3))

### Visualizar predicciones

Mostramos 15 imágenes del conjunto de prueba con la predicción del modelo. El título aparece en **verde** si la predicción es correcta y en **rojo** si es un error (indicando también la etiqueta real).

In [ ]:
# Mostrar 15 predicciones (verde = acierto, rojo = error)
plt.figure(figsize=(12, 6))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')
    correcto = y_pred[i] == y_true[i]
    color = 'green' if correcto else 'red'
    titulo = f'Pred: {y_pred[i]}' if correcto else f'Pred: {y_pred[i]} (real {y_true[i]})'
    plt.title(titulo, color=color, fontsize=10)
    plt.axis('off')
plt.tight_layout()
plt.show()

### Comparativa de los tres modelos

Resumimos la precisión en el conjunto de prueba de los tres modelos construidos para ver el efecto de añadir capas ocultas y regularización.

In [ ]:
import pandas as pd

modelos = {
    'model  (1 capa, softmax)': model,
    'model2 (2 capas ocultas ReLU)': model2,
    'model3 (2 capas + dropout)': model3,
}

filas = []
for nombre, m in modelos.items():
    loss, acc = m.evaluate(X_test, y_test, verbose=0)
    filas.append({'Modelo': nombre, 'Loss (test)': round(loss, 4), 'Accuracy (test)': round(acc, 4)})

pd.DataFrame(filas)